# 03_4 · Đánh giá cuối trên tập test, phân tích và lưu mô hình
**Phần 4/4.** Dùng cấu hình đã chọn ở 03_3, huấn luyện lại với dữ liệu đến 30/11 và đánh giá trên tháng 12 **một lần duy nhất**. Sau đó kiểm định chênh lệch, phân tích thêm, xem ví dụ cụ thể và lưu mô hình triển khai.

**Đầu ra:** `reports/recsys_ket_qua_test.csv`, `reports/figs/R1_so_sanh_mo_hinh.png`, `models/recsys_final.pkl`.

In [ ]:
from recsys_lib import *
from IPython.display import display

inter, item_stats, catalog, n_inv_total = load_cache()
st = load_state(); blacklist = set(st["blacklist"])
print(f"{len(inter):,} dòng | loại {len(blacklist)} mã không phải hàng hóa")
best = pd.Series({"mo_hinh": st["best_config"]}); cold_best = st["cold_choice"]
val = Split(inter, catalog, blacklist, *VAL)

## 7. Đánh giá cuối trên tập test (tháng 12) — chạy một lần
Tham số cố định theo kết quả validation; huấn luyện lại với dữ liệu đến 30/11.

In [ ]:
test = Split(inter, catalog, blacklist, *TEST); print(test.summary())
print("Cấu hình:", best.mo_hinh, "| khách mới dùng:", cold_best)
cold_model_t = make_cold(cold_best, test)
final_model = rebuild(best.mo_hinh, test); pop_t = Popularity(30).fit(test)
test_res = {}
entries = [("Baseline: bán chạy 30 ngày (người mua)", pop_t, pop_t),
           ("Bán chạy theo giới tính", SegmentPopularity(30, 0.7).fit(test), cold_model_t),
           (f"Mô hình tốt nhất: {best.mo_hinh}", final_model, cold_model_t)]
frames = []
for name, mdl, cm in entries:
    r = evaluate(test, make_scorer(mdl, cm)); test_res[name] = r; frames.append(summarize(r, name))
final_tab = pd.concat(frames)
final_tab.to_csv(REPORTS / "recsys_ket_qua_test.csv", index=False)
final_tab[final_tab.nhom.isin(["tat_ca", "co_lich_su", "khach_moi"])][["mo_hinh", "nhom", "so_khach", f"recall@{K}", f"ndcg@{K}", f"map@{K}", f"hit_rate@{K}", "coverage", "da_dang_nganh"]]

In [ ]:
# Chênh lệch so với baseline có ý nghĩa không? (bootstrap ghép cặp, 95%)
names = list(test_res); b0, b1 = test_res[names[0]], test_res[names[-1]]
for g, mask in [("co_lich_su", test.known), ("khach_moi", ~test.known), ("tat_ca", np.ones(len(test.known), bool))]:
    for metric in ["ndcg", "recall"]:
        d, lo, hi = paired_bootstrap(getattr(b1, metric)[mask], getattr(b0, metric)[mask])
        print(f"{g:11s} {metric:6s}@{K}: mô hình tốt nhất − baseline = {d:+.4f} (95% CI {lo:+.4f} đến {hi:+.4f})",
              "→ khác biệt có ý nghĩa" if lo > 0 or hi < 0 else "→ chưa khác biệt rõ ràng")
print(f"\nTrần Recall (mẫu còn bán ở kỳ test): {test.ceiling:.3f}")

In [ ]:
# Biểu đồ so sánh mô hình trên tập test
plot = final_tab[final_tab.nhom.isin(["co_lich_su", "khach_moi"])].pivot(index="mo_hinh", columns="nhom", values=f"recall@{K}").loc[[e[0] for e in entries]]
labels = ["Baseline\n(bán chạy 30 ngày)", "Bán chạy\ntheo giới tính", "Mô hình tốt nhất\n(ItemKNN + giới tính)" if "Blend" in best.mo_hinh else "Mô hình\ntốt nhất"]
fig, ax = plt.subplots(figsize=(9, 4.5)); x = np.arange(len(plot)); wd = 0.38
b1 = ax.bar(x - wd/2, plot["co_lich_su"], wd, label="Khách có lịch sử", color="#2a6f97")
b2 = ax.bar(x + wd/2, plot["khach_moi"], wd, label="Khách mới", color="#e07a5f")
for b in list(b1) + list(b2): ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.001, f"{b.get_height():.3f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, plot.values.max() * 1.25)
gain = plot["co_lich_su"].iloc[-1] / max(plot["co_lich_su"].iloc[0], 1e-9)
ax.set_title(f"Với khách có lịch sử, mô hình tốt nhất đạt Recall@{K} gấp {gain:.2f} lần baseline")
ax.set_ylabel(f"Recall@{K} (tháng 12/2025)"); ax.legend(loc="upper right", frameon=False, ncol=2)
plt.savefig(FIGS / "R1_so_sanh_mo_hinh.png", dpi=160, bbox_inches="tight"); plt.show()

## 8. Phân tích thêm (trên validation, không dùng test)

In [ ]:
# (a) Có nên loại mẫu đã mua khỏi gợi ý? (b) Cửa sổ "mẫu còn bán" 30/60/90 ngày ảnh hưởng thế nào?
rows = []
mdl = rebuild(best.mo_hinh, val); first60 = FirstTimer(60).fit(val)
for ex in [True, False]:
    r = evaluate(val, make_scorer(mdl, first60), exclude_seen=ex)
    d = summarize(r, f"{best.mo_hinh} | loại mẫu đã mua={ex}"); rows.append(d[d.nhom == "co_lich_su"])
for days in [30, 60, 90]:
    sp_ = Split(inter, catalog, blacklist, *VAL, active_days=days)
    m = rebuild(best.mo_hinh, sp_); r = evaluate(sp_, make_scorer(m, FirstTimer(60).fit(sp_)))
    d = summarize(r, f"{best.mo_hinh} | cửa sổ mẫu còn bán={days} ngày", {"tran_recall": sp_.ceiling}); rows.append(d[d.nhom == "co_lich_su"])
pd.concat(rows)[["mo_hinh", f"recall@{K}", f"ndcg@{K}", "coverage", "tran_recall"]]

In [ ]:
# Ví dụ gợi ý cụ thể cho một khách có lịch sử trong tập test (kiểm tra bằng mắt có hợp lý không)
i = int(np.where(test.known2)[0][0]); u = test.eval_users[i]
bought = test.train[test.train.customer_id == u].sort_values("ngay").tail(5)
print("Khách", u, "đã mua gần đây:"); display(bought.assign(ten=bought.item.map(dict(zip(catalog.item, catalog.product_name))))[["ngay", "item", "ten"]])
sc = make_scorer(final_model, cold_model_t)(test.rows[i:i+1])[0]
sc = np.where(test.seen[test.rows[i]].toarray().ravel() > 0, -1e9, sc); top = np.argsort(-sc)[:K]
print("Gợi ý top-10:"); display(pd.DataFrame({"item": test.items[top], "ten": test.name[top], "nganh": test.group[top]}))
print("Thực tế khách mua trong tháng 12:", sorted(set(inter[(inter.customer_id == u) & (inter.ngay >= test.start)].item)))

## 9. Lưu mô hình và hàm gợi ý
Mô hình dùng để triển khai được huấn luyện lại trên **toàn bộ dữ liệu 2025** (không còn tập kiểm tra).

In [ ]:
deploy = Split(inter, catalog, blacklist, "2026-01-01", "2026-02-01")
dep_model = rebuild(best.mo_hinh, deploy); dep_cold = FirstTimer(60).fit(deploy)
dep_scorer = make_scorer(dep_model, dep_cold)

def recommend(customer_id, k=K, exclude_seen=True):
    # Gợi ý top-k mẫu cho khách; khách chưa có lịch sử nhận gợi ý của khách mới
    row = deploy.user_idx.get(str(customer_id), -1)
    s = dep_scorer(np.array([row]))[0].copy()
    if exclude_seen and row >= 0:
        s[deploy.seen[row].toarray().ravel() > 0] = -1e9
    top = np.argsort(-s)[:k]
    return pd.DataFrame({"item": deploy.items[top], "ten_san_pham": deploy.name[top], "nganh_hang": deploy.group[top], "diem": s[top]})

with open(MODELS / "recsys_final.pkl", "wb") as f:
    pickle.dump({"cau_hinh": best.mo_hinh, "items": deploy.items, "users": deploy.users,
                 "blacklist": sorted(blacklist), "k": K}, f)
recommend(deploy.users[0])

## Việc cần làm sau khi chạy xong cả 4 phần
1. Điền kết quả vào báo cáo: bảng baseline (03_2), bảng validation (03_3), bảng test và khoảng tin cậy (03_4), biểu đồ `R1_so_sanh_mo_hinh.png`.
2. Viết phân tích lỗi: mô hình hơn baseline bao nhiêu với khách có lịch sử, vì sao khách mới chỉ dùng được độ phổ biến, trần Recall nói gì.
3. Gửi tôi các file CSV trong `reports/recsys_*.csv` để tôi giúp viết phần kết quả và thảo luận.